### Prompt Drift

Unlike a bug in your code that throws an error, prompt drift is a gradual shift where the same input results in a different or degraded output over time.

#### 1. The "Ghost" Update (Provider Drift)

The most common cause is when model providers (OpenAI, Anthropic, etc.) update their models behind the same API endpoint. Even if the version name (like gpt-4o) stays the same, the provider might tweak the "alignment" (safety layers) or "quantization" (efficiency).

#### 2. Context Inflation (RAG Drift)

As your database grows, the type of data being injected changes. If you originally tuned your prompt for short FAQ snippets, but your system starts retrieving long, complex legal documents, the model's focus will "drift" toward the new data density, often ignoring your original formatting instructions.

#### 3. Semantic Saturation & Cultural Drift

Language itself evolves. If your prompt uses specific slang, professional terminology, or instructions based on a 2023 world-view, its effectiveness will decay as the model's "internal world map" is updated with more recent data during its own training cycles.

### Issues with prompt Drift

Cascading Failure: If you have multiple prompts working in a sequence (Prompt A $\rightarrow$ Prompt B), Prompt B, which expects a concise summary, gets confused by the extra text and starts hallucinating

### Model degradation

##### 1. Concept Drift (Covariate Shift)

A model provides outdated information because it hasn't been trained on recent. So the model gives a confident answer that was true a year ago but is wrong today (a "Temporal Hallucination").

Eg: An LLM might correctly identify a question about "travel requirements" but provide answers that were true three months ago but are now legally incorrect due to a policy change.

##### 2. Model Collapse (Synthetic Data Poisoning)

A unique risk for LLMs is Model Collapse. This happens when AI models are trained on data that was itself generated by other AI (synthetic data) rather than humans.

Information Entropy: Just like a photocopy of a photocopy, each generation loses the "long-tail" or rare data points that make human language rich.

Homogenization: The model becomes "boring," repetitive, and eventually nonsensical because it starts echoing its own mistakes and biases until it "collapses" into a narrow set of predictable responses.

##### 3. Operational & Infrastructure Factors

Providers may "quantize" (shrink) a model to save money on servers. While this makes it faster, it can cause a subtle decline in its ability to handle complex reasoning or rare edge cases.

### How to detect

- Prompt Versioning: Always link your traces to a specific Prompt Version. If you see "Avg. Score" drop for v2.1 compared to last week, you’ve found drift.

- Embedding Shifts: Monitor the "semantic distance" of your outputs. If the vector representations of your answers start moving away from your "Golden Dataset," the prompt is losing its grip.

- Verbosity Monitoring: A sudden spike in completion_tokens for the same prompt is a classic sign of a provider-side update making the model more talkative.

- Maintain a Golden Test Dataset: Keep a fixed set of prompts and test them regularly to check if the output quality changes.

In [ ]:
from langfuse import Langfuse
from langfuse.openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()

langfuse = Langfuse()

LITELLM_API_KEY = os.getenv("LITELLM_API_KEY")
LITELLM_BASE_URL = os.getenv("LITELLM_BASE_URL")
client = OpenAI(
    api_key=LITELLM_API_KEY,
    base_url=LITELLM_BASE_URL,
)

def description(topic):
    prompt = langfuse.get_prompt("description", version=1)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt.compile(topic=topic)}],
        langfuse_prompt=prompt
    )
    
    return response.choices[0].message.content

print(description("Dark Matter"))

### References

- https://cobusgreyling.medium.com/prompt-drift-4873f37c43c8
- https://www.organizing4innovation.com/ai-model-drift-what-it-is-and-what-it-isnt/